# 03. Propensity: кто купит партнёрский продукт (cross-sell)

**Цель.** Для каждого клиента оценить вероятность, что он **впервые купит** выбранный партнёрский продукт (по умолчанию авиабилеты `flights`) в ближайшие 30 дней, чтобы направлять предложения на тех, кто с наибольшей вероятностью откликнется.

**Как устроено.**
1. Нормализуем названия продуктов (`Тикетон` и `ticketon` это один продукт).
2. Кандидаты: клиенты, активные за 60 дней до отсечки и **не покупавшие этот продукт ранее**.
3. Цель: первая покупка продукта в следующие 30 дней (1) или нет (0).
4. Признаки считаем только по данным до отсечки: поведение в транзакциях и покупки у партнёров.
5. Обучаем на отсечке 13 марта, проверяем на отсечке 12 апреля, на **непересекающихся группах клиентов**.
6. Оцениваем AUC, PR-AUC и lift (событие редкое, около 2%, поэтому accuracy бесполезна).
7. Проверяем, сколько сигнала даёт история покупок у партнёров, и объясняем модель через SHAP.

**Допущения:**
- Успешными транзакциями считаем `Успешные`, `AUSR0401`, `AUSR0402`.
- «Не купил» означает «не купил через приложение»: покупки вне SuperApp мы не видим.
- Одинаковые продукты с разными названиями объединены правилами из словаря `PRODUCT_MAP`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TX_FILE = DATA_DIR / "SAPP транзакции.csv"
PP_FILE = DATA_DIR / "SAPP Покупки у партнеров.csv"

SUCCESS_STATUSES = ("Успешные", "AUSR0401", "AUSR0402")
LOOKBACK_DAYS = 60
HORIZON_DAYS = 30

PRODUCT = "flights"   # <- меняй: ticketon, train_tickets, groceries, fmedia, telecom

# Приводим разные написания к одному названию продукта
PRODUCT_MAP = {
    "ticketon": "ticketon", "Тикетон": "ticketon",
    "flights": "flights",
    "train_tickets": "train_tickets",
    "groceries": "groceries",
    "fmedia": "fmedia",
    "telecom": "telecom", "ftelecom": "telecom", "Freedom Mobile": "telecom", "mobile": "telecom",
}
PRODUCTS = sorted(set(PRODUCT_MAP.values()))

con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")
print("OK:", TX_FILE.exists(), PP_FILE.exists(), "| продукт:", PRODUCT)

## 1. Очистка данных

Транзакции чистим так же, как в churn-ноутбуке. Покупки у партнёров: нормализуем названия продуктов, суммы берём по модулю и обрезаем выбросы.

In [ ]:
status_list = ", ".join(f"'{s}'" for s in SUCCESS_STATUSES)

con.execute(f"""
    CREATE OR REPLACE TABLE tx_raw AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id,
           CAST(transaction_date AS DATE) AS d,
           operation_type,
           ABS(CAST(transaction_sum AS DOUBLE)) AS amt,
           terminal_type
    FROM read_csv_auto('{TX_FILE.as_posix()}')
    WHERE transaction_status IN ({status_list}) AND customer_id IS NOT NULL
""")
cap = con.execute("SELECT quantile_cont(amt, 0.999) FROM tx_raw").fetchone()[0]
con.execute(f"""
    CREATE OR REPLACE TABLE tx AS
    SELECT customer_id, d, operation_type, LEAST(amt, {cap}) AS amt, terminal_type FROM tx_raw
""")

case_sql = "CASE app_name " + " ".join(
    f"WHEN '{k}' THEN '{v}'" for k, v in PRODUCT_MAP.items()) + " ELSE 'other' END"

con.execute(f"""
    CREATE OR REPLACE TABLE pp_raw AS
    SELECT CAST(customer_id AS BIGINT) AS customer_id,
           CAST(purchase_date AS DATE) AS d,
           {case_sql} AS prod,
           ABS(CAST(purchase_amount AS DOUBLE)) AS amt,
           ABS(CAST(cashback_amount AS DOUBLE)) AS cashback
    FROM read_csv_auto('{PP_FILE.as_posix()}')
    WHERE customer_id IS NOT NULL
""")
pp_cap = con.execute("SELECT quantile_cont(amt, 0.999) FROM pp_raw").fetchone()[0]
con.execute(f"""
    CREATE OR REPLACE TABLE pp AS
    SELECT customer_id, d, prod, LEAST(amt, {pp_cap}) AS amt, cashback FROM pp_raw
""")

print("Транзакции:", con.execute("SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM tx").fetchone())
print(con.execute("""
    SELECT prod, COUNT(*) AS purchases, COUNT(DISTINCT customer_id) AS customers,
           MIN(d) AS first_day, MAX(d) AS last_day
    FROM pp GROUP BY prod ORDER BY purchases DESC
""").df())

## 2. Даты отсечки

Те же две отсечки, что и в churn: обучение на `CUTOFF_TRAIN`, проверка на более позднем `CUTOFF_TEST`.

In [ ]:
last_day = pd.Timestamp(con.execute("SELECT MAX(d) FROM tx").fetchone()[0])
CUTOFF_TEST = last_day - pd.Timedelta(days=HORIZON_DAYS)
CUTOFF_TRAIN = CUTOFF_TEST - pd.Timedelta(days=HORIZON_DAYS)
print("Отсечка train:", CUTOFF_TRAIN.date(), "| Отсечка test:", CUTOFF_TEST.date())

## 3. Признаки и целевая переменная

**Кандидаты:** активны в транзакциях за последние 60 дней и ещё не покупали выбранный продукт.
**Цель:** первая покупка продукта в следующие 30 дней.
**Признаки:** поведение в транзакциях и история покупок у партнёров (сколько, каких продуктов, на какие суммы, как давно).

In [ ]:
def build_dataset(cutoff: pd.Timestamp, product: str) -> pd.DataFrame:
    c = cutoff.strftime("%Y-%m-%d")
    prod_cols = ",\n            ".join(
        f"COUNT(*) FILTER (WHERE prod = '{p}') AS pp_{p}" for p in PRODUCTS)
    prod_select = ",\n        ".join(f"COALESCE(pf.pp_{p}, 0) AS pp_{p}" for p in PRODUCTS)

    q = f"""
    WITH base AS (
        SELECT DISTINCT customer_id
        FROM tx
        WHERE d <= DATE '{c}' AND d > DATE '{c}' - {LOOKBACK_DAYS}
          AND customer_id NOT IN (
              SELECT customer_id FROM pp WHERE prod = '{product}' AND d <= DATE '{c}')
    ),
    f AS (
        SELECT
            t.customer_id,
            date_diff('day', MAX(d), DATE '{c}')                      AS recency_days,
            date_diff('day', MIN(d), DATE '{c}')                      AS tenure_days,
            COUNT(*)                                                   AS n_tx,
            COUNT(DISTINCT d)                                          AS n_active_days,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 7)                 AS n_tx_7d,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 14)                AS n_tx_14d,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 30)                AS n_tx_30d,
            COUNT(*) FILTER (WHERE d <= DATE '{c}' - 30
                               AND d >  DATE '{c}' - 60)               AS n_tx_prev30d,
            COALESCE(SUM(amt) FILTER (WHERE d > DATE '{c}' - 30), 0)   AS sum_amt_30d,
            AVG(amt)                                                   AS avg_amt,
            AVG((operation_type = 'Purchase')::INT)                    AS share_purchase,
            AVG((operation_type = 'P2P Debit')::INT)                   AS share_p2p_debit,
            AVG((operation_type = 'P2P Credit')::INT)                  AS share_p2p_credit,
            AVG((operation_type = 'ATM Cash withdrawal')::INT)         AS share_atm,
            AVG((terminal_type = 'ePOS')::INT)                         AS share_epos
        FROM tx t JOIN base b USING (customer_id)
        WHERE t.d <= DATE '{c}'
        GROUP BY t.customer_id
    ),
    pf AS (
        SELECT
            customer_id,
            COUNT(*)                                       AS pp_n,
            COUNT(*) FILTER (WHERE d > DATE '{c}' - 30)    AS pp_n_30d,
            COUNT(DISTINCT prod)                           AS pp_n_products,
            SUM(amt)                                       AS pp_sum_amt,
            SUM(cashback)                                  AS pp_cashback_sum,
            date_diff('day', MAX(d), DATE '{c}')           AS pp_recency_days,
            {prod_cols}
        FROM pp
        WHERE d <= DATE '{c}'
        GROUP BY customer_id
    ),
    y AS (
        SELECT DISTINCT customer_id, 1 AS target
        FROM pp
        WHERE prod = '{product}' AND d > DATE '{c}' AND d <= DATE '{c}' + {HORIZON_DAYS}
    )
    SELECT f.*,
        COALESCE(pf.pp_n, 0)            AS pp_n,
        COALESCE(pf.pp_n_30d, 0)        AS pp_n_30d,
        COALESCE(pf.pp_n_products, 0)   AS pp_n_products,
        COALESCE(pf.pp_sum_amt, 0)      AS pp_sum_amt,
        COALESCE(pf.pp_cashback_sum, 0) AS pp_cashback_sum,
        COALESCE(pf.pp_recency_days, 999) AS pp_recency_days,
        {prod_select},
        COALESCE(y.target, 0)           AS target
    FROM f
    LEFT JOIN pf USING (customer_id)
    LEFT JOIN y  USING (customer_id)
    """
    df = con.execute(q).df()
    df["ratio_30d_vs_prev30d"] = df["n_tx_30d"] / (df["n_tx_prev30d"] + 1)
    df["tx_per_active_day"] = df["n_tx"] / df["n_active_days"].clip(lower=1)
    return df

train_df = build_dataset(CUTOFF_TRAIN, PRODUCT)
test_df = build_dataset(CUTOFF_TEST, PRODUCT)

for name, d in [("train", train_df), ("test", test_df)]:
    print(f"{name}: {len(d):,} кандидатов, покупателей {int(d['target'].sum()):,} "
          f"({d['target'].mean():.2%})")
train_df.head()

## 4. Непересекающиеся клиенты в обучении и проверке

Один и тот же клиент мог бы попасть и в обучение, и в проверку (он активен в обоих периодах), и модель частично «запоминала» бы клиентов. Поэтому мы случайно делим всех клиентов на две группы: на группе A обучаем (отсечка train), на группе B проверяем (отсечка test).

In [ ]:
all_ids = np.unique(np.concatenate([train_df["customer_id"], test_df["customer_id"]]))
rng = np.random.RandomState(SEED)
group_a = set(rng.choice(all_ids, size=len(all_ids) // 2, replace=False))

train_part = train_df[train_df["customer_id"].isin(group_a)].reset_index(drop=True)
test_part = test_df[~test_df["customer_id"].isin(group_a)].reset_index(drop=True)

print("Пересечение клиентов:", len(set(train_part.customer_id) & set(test_part.customer_id)))
print(f"train: {len(train_part):,} (покупателей {int(train_part.target.sum()):,}, "
      f"{train_part.target.mean():.2%})")
print(f"test:  {len(test_part):,} (покупателей {int(test_part.target.sum()):,}, "
      f"{test_part.target.mean():.2%})")

## 5. Модели и метрики

Событие редкое, поэтому смотрим не только AUC, но и **PR-AUC** (насколько хорошо модель собирает покупателей наверху списка) и **lift** в верхних 10% и 20% клиентов. Для сравнения, у случайного выбора PR-AUC примерно равна доле покупателей.

In [ ]:
ALL_FEATURES = [c for c in train_part.columns if c not in ("customer_id", "target")]
# Убираем константные признаки (например, покупки самого целевого продукта = 0 у всех кандидатов)
ALL_FEATURES = [c for c in ALL_FEATURES if train_part[c].nunique() > 1]
PP_FEATURES = [c for c in ALL_FEATURES if c.startswith("pp_")]
BEHAV_FEATURES = [c for c in ALL_FEATURES if c not in PP_FEATURES]

def metrics(y, p):
    y = np.asarray(y)
    base = y.mean()
    out = {"AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
           "Доля покупателей": base}
    for share in (0.10, 0.20):
        k = int(len(p) * share)
        idx = np.argsort(-p)[:k]
        out[f"Precision@top{int(share*100)}%"] = y[idx].mean()
        out[f"Lift@top{int(share*100)}%"] = y[idx].mean() / base
        out[f"Recall@top{int(share*100)}%"] = y[idx].sum() / y.sum()
    return out

def make_models():
    return {
        "LogisticRegression": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            LogisticRegression(max_iter=1000)),
        "LightGBM": LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
            subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostClassifier(
            iterations=300, learning_rate=0.08, depth=6,
            random_seed=SEED, verbose=0, thread_count=2),
    }

def log_for_linear(X):
    X = X.copy()
    for c in X.columns:
        if X[c].min() >= 0 and X[c].max() > 50:
            X[c] = np.log1p(X[c])
    return X

def fit_eval(features, tr, te, only=None):
    res, preds, fitted = {}, {}, {}
    for name, model in make_models().items():
        if only and name not in only:
            continue
        Xtr, Xte = tr[features], te[features]
        if name == "LogisticRegression":
            Xtr, Xte = log_for_linear(Xtr), log_for_linear(Xte)
        model.fit(Xtr, tr["target"])
        p = model.predict_proba(Xte)[:, 1]
        res[name], preds[name], fitted[name] = metrics(te["target"], p), p, model
    return pd.DataFrame(res).T, preds, fitted

results, preds, fitted = fit_eval(ALL_FEATURES, train_part, test_part)
results.round(3)

## 6. Сколько сигнала даёт история покупок у партнёров

Сравниваем модель только на поведении в транзакциях и модель с историей партнёрских покупок. Если прирост большой, то кросс-продажа действительно опирается на то, что клиент уже покупает у партнёров.

In [ ]:
res_beh, _, _ = fit_eval(BEHAV_FEATURES, train_part, test_part, only=["LightGBM"])
cmp = pd.DataFrame({
    "Только транзакции": res_beh.loc["LightGBM", ["AUC", "PR-AUC", "Lift@top10%"]],
    "Транзакции + партнёры": results.loc["LightGBM", ["AUC", "PR-AUC", "Lift@top10%"]],
}).round(3)
cmp["Прирост"] = (cmp["Транзакции + партнёры"] - cmp["Только транзакции"]).round(3)
cmp

## 7. Lift по децилям

Сортируем клиентов теста по вероятности покупки и делим на 10 групп. Показываем долю реальных покупателей в каждой группе и во сколько раз она выше средней.

In [ ]:
best_name = results["PR-AUC"].idxmax()
print("Лучшая модель по PR-AUC:", best_name)

p = preds[best_name]
d = pd.DataFrame({"p": p, "target": test_part["target"].values})
d["decile"] = 10 - pd.qcut(d["p"].rank(method="first"), 10, labels=False)
base_rate = d["target"].mean()
lift = (d.groupby("decile")
          .agg(clients=("target", "size"), buyers=("target", "sum"), buy_rate=("target", "mean"))
          .assign(lift=lambda x: x["buy_rate"] / base_rate))
lift["cum_buyers_share"] = d.groupby("decile")["target"].sum().cumsum() / d["target"].sum()
print(f"Средняя доля покупателей: {base_rate:.3%}")
lift.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(lift.index, lift["buy_rate"] * 100)
ax.axhline(base_rate * 100, linestyle="--", color="gray", label="Средняя доля покупателей")
ax.set_xlabel("Дециль (1 = самая высокая вероятность покупки)")
ax.set_ylabel("Доля купивших, %")
ax.set_title(f"Покупатели {PRODUCT} по децилям ({best_name})")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Объяснимость: SHAP

Какие признаки повышают или понижают вероятность первой покупки.

In [ ]:
import shap

tree_model = fitted["CatBoost"]   # дерево-модель с лучшим AUC
sample = test_part[ALL_FEATURES].sample(min(20000, len(test_part)), random_state=SEED)
sv = shap.TreeExplainer(tree_model).shap_values(sample)
sv = sv[1] if isinstance(sv, list) else sv

shap.summary_plot(sv, sample, show=False)
plt.tight_layout()
plt.show()

shap.summary_plot(sv, sample, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

## 9. Другие продукты

Тот же подход для нескольких продуктов (только LightGBM, чтобы было быстро). Показывает, где cross-sell предсказуем лучше всего.

In [ ]:
summary = []
for prod in ["ticketon", "flights", "train_tickets", "groceries"]:
    tr = build_dataset(CUTOFF_TRAIN, prod)
    te = build_dataset(CUTOFF_TEST, prod)
    ids = np.unique(np.concatenate([tr["customer_id"], te["customer_id"]]))
    ga = set(np.random.RandomState(SEED).choice(ids, size=len(ids) // 2, replace=False))
    tr = tr[tr["customer_id"].isin(ga)].reset_index(drop=True)
    te = te[~te["customer_id"].isin(ga)].reset_index(drop=True)
    feats = [c for c in tr.columns if c not in ("customer_id", "target") and tr[c].nunique() > 1]
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                       subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbose=-1)
    m.fit(tr[feats], tr["target"])
    r = metrics(te["target"], m.predict_proba(te[feats])[:, 1])
    summary.append({"Продукт": prod, "Покупателей в тесте": int(te["target"].sum()),
                    "Доля": r["Доля покупателей"], "AUC": r["AUC"], "PR-AUC": r["PR-AUC"],
                    "Lift@top10%": r["Lift@top10%"], "Recall@top20%": r["Recall@top20%"]})
pd.DataFrame(summary).round(3)

## 10. Бизнес-применение

Что даёт рассылка предложения верхним 10% клиентов по сравнению со случайной рассылкой. **Параметры ниже это допущения**: подставь реальные значения или говори о них как о гипотезах для A/B-теста.

In [ ]:
TOP_SHARE = 0.10
COST_PER_CONTACT = 100        # допущение, тенге
OFFER_RESPONSE_LIFT = 0.20    # допущение: предложение повышает вероятность покупки на 20% относительно базы
VALUE_PER_PURCHASE = 5000     # допущение: ценность одной покупки для компании, тенге

n = len(d)
k = int(n * TOP_SHARE)
top = d.sort_values("p", ascending=False).head(k)

model_buyers = top["target"].sum()                  # покупатели, попавшие в топ
random_buyers = k * base_rate                        # покупатели при случайной рассылке тому же числу клиентов
print(f"Контактируем: {k:,} клиентов ({TOP_SHARE:.0%} кандидатов)")
print(f"Из них купят сами (без нашего воздействия): {model_buyers:,.0f} "
      f"({model_buyers / k:.2%}) против {random_buyers:,.0f} при случайном выборе "
      f"({base_rate:.2%})")
print(f"Охват всех будущих покупателей: {model_buyers / d['target'].sum():.1%}")

extra = model_buyers * OFFER_RESPONSE_LIFT
cost = k * COST_PER_CONTACT
print(f"Допущение: предложение даёт +{OFFER_RESPONSE_LIFT:.0%} к покупкам -> "
      f"{extra:,.0f} дополнительных покупок")
gain = extra * VALUE_PER_PURCHASE
print(f"Затраты: {cost:,.0f} тг | Выгода: {gain:,.0f} тг | Результат: {gain - cost:,.0f} тг")
print(f"Точка безубыточности: ценность одной покупки должна быть не меньше {cost / extra:,.0f} тг "
      f"при стоимости контакта {COST_PER_CONTACT} тг")
print("Результат целиком зависит от допущений, поэтому его нужно проверить A/B-тестом.")

## 11. Выводы (заполни своими словами по результатам)

1. **Постановка:** предсказываем первую покупку продукта `...` в следующие 30 дней среди активных клиентов, которые его ещё не покупали. Доля покупателей около ...
2. **Качество:** AUC = ..., PR-AUC = ... (у случайной модели PR-AUC равна доле покупателей ...). Lift в верхних 10% равен ...
3. **Роль истории партнёрских покупок:** добавление признаков покупок у партнёров меняет AUC на ... и lift на ..., значит, ...
4. **Что влияет по SHAP:** главные признаки ...
5. **Продукты:** лучше всего предсказуем ..., хуже ...
6. **Рекомендация:** предлагать продукт верхним 10–20% по вероятности, сегментировать по похожим покупкам, проверить эффект A/B-тестом с контрольной группой.
7. **Ограничения:** около 4,5 месяцев данных; покупки вне приложения не видны; модель показывает склонность к покупке, а не причинный эффект предложения (для этого нужен uplift-подход или A/B-тест).